In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt

csv_path = os.path.join(path, "Q3_data.csv")
df = pd.read_csv(csv_path)

In [ ]:
# Task 2: Write your code here:
df.head()


In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 1: Write your code here:
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")
  return missing_values[missing_values > 0]

missing_values = check_missing_values(df)
ms = (list(missing_values.index))
# fill all of them with the mean
df[ms] = df[ms].fillna(df[ms].mean())

In [ ]:
# Task 2: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 3: Write your code here:
from sklearn.preprocessing import LabelEncoder

categorical_cols = df.select_dtypes(include=["object"]).columns
print("Categorical Columns:", list(categorical_cols))

for col in categorical_cols:
  le = LabelEncoder()
  df[col] = le.fit_transform(df[col])

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import StandardScaler

features = df.columns.drop("Target")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df[features] = scaler.fit_transform(df[features])
df.head()

In [ ]:
# Task 5: Write your code here:
import seaborn as sns
def check_target_imbalance(df, target_column):
  print("Target Distribution:")
  print(df[target_column].value_counts(normalize=True))
  sns.countplot(x=df[target_column])
  plt.title("Target Distribution")
  plt.show()

check_target_imbalance(df, "Target")

## target is imbalanced for what we see

In [ ]:
# Task 1: Write your code here:
X = df.drop("Target", axis=1).astype(float)
y = df['Target'].astype(float)

In [ ]:
!pip install catboost

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from catboost import CatBoostClassifier

model = CatBoostClassifier(
      verbose=0,
      n_estimators=320,
      max_depth=4
  )
n_splits = 5
kf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
for fold_idx, (train_index, test_index) in enumerate(kf.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]


  model.fit(X_train, y_train)
  y_pred = model.predict(X_test)
  print(f"f1: {f1_score(y_test, y_pred)}")

print(f"\n\nTotal f1: {f1_score(y_test, y_pred)}")

In [ ]:
# Task 1: Write your code here:
feature_cols = df.columns.drop("Target")

feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 30))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
most_impo = feature_importance["feature"][0]
print(f"the most important feature is {most_impo}")

In [ ]:
# Task Bonus: Write your code here:
X = df[most_impo]
y = df["Target"]

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from catboost import CatBoostClassifier

model = CatBoostClassifier(
      verbose=0,
      n_estimators=320,
      max_depth=4
  )
n_splits = 5
kf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
for fold_idx, (train_index, test_index) in enumerate(kf.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]


  model.fit(X_train, y_train)
  y_pred = model.predict(X_test)
  print(y_test.shape, y_pred.shape)
  print(X_test.shape)
  print(f"f1: {f1_score(y_test, y_pred)}")

print(f"\n\nTotal f1: {f1_score(y_test, y_pred)}")